# E207 Final Project: Fast Music Retrieval Under Time Warping

K-means clustering method

Collaborators: Jordan Carlin, Jasper Cox, Thomas Lilygren

In [1]:
%matplotlib inline

In [2]:
import numpy as np
import librosa as lb
import matplotlib.pyplot as plt
import IPython.display as ipd
import scipy.io as sio
import scipy.signal
import glob
import os.path
import subprocess
import pickle
import joblib
import difflib
import string
import soundfile as sf
from pathlib import Path
from typing import Literal
from sklearn.cluster import MiniBatchKMeans as miniK
from sklearn.feature_extraction.text import TfidfTransformer

In [3]:
# Configuration
N_FFT = 2048
HOP_LENGTH = 512
EPS = np.finfo(np.float64).eps  # small constant to avoid division by zero
RANK = 4
LETTERS = list(string.ascii_lowercase)[0:RANK]
TARGET_SR = 22050
K_VAL = 256 # used for grouping sounds in a particular file

In [4]:
def load_audio(audio_file: Path) -> tuple[np.ndarray, int | float]:
    """
    Load an audio file. Returns a tuple of the audio time series and the sample rate.
    """
    audio, sr = lb.load(audio_file, sr=TARGET_SR)
    return audio, sr

In [5]:
def time_to_freq(audio: np.ndarray, sr: int | float, algorithim: Literal["stft", "cqt", "logmel", "chroma"] = "stft", n_fft: int = N_FFT, hop_length: int = HOP_LENGTH) -> np.ndarray:
    """
    Convert a time-domain audio signal to a frequency-domain representation.
    """
    match algorithim:
        case "stft":
            X = lb.stft(audio, n_fft=n_fft, hop_length=hop_length)
            return X
        case "cqt":
            C = lb.cqt(audio, sr=sr, hop_length=hop_length)
            return C
        case "logmel":
            S = lb.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=128)
            log_S = lb.power_to_db(S, ref=np.max)
            return log_S
        case "chroma":
            C = lb.feature.chroma_stft(y=audio, sr=sr, n_fft=n_fft, hop_length=hop_length)
            return C
        case _:
            raise ValueError(f"Unsupported type: {algorithim}")

In [8]:
def test_single_file(audio_file: Path, n_components=RANK, k=K_VAL, sr=TARGET_SR, max_frames=1000) -> None:
    
    # Get frequency conversion matrix
    audio, sr = load_audio(audio_file)
    X = time_to_freq(audio, sr, "stft")
    X = np.abs(X).T

    # Apply mini K means protocol
    kmeans = miniK(n_clusters=k, batch_size=4096, n_init=3, max_iter=100, random_state=42, verbose=1)
    kmeans.fit(X)

    # Quantize file - assign frame to nearest centroid, represent recording as codewords
    file_codes = kmeans.predict(X)

    #print(f"Quantized: {len(file_codes)} frames")
    #print(f"Unique codewords used: {len(np.unique(file_codes))} / {k}")
    return file_codes

In [ ]:
def check_match(file1, file2, k=K_VAL, sr=TARGET_SR):

    # Get file codes for first file
    file1_codes = test_single_file(file1, k, sr)

    # Hist for file 1 to fit tfidf transformer
    file1_hist = np.bincount(file1_codes, minlength=k).reshape(1,-1)

    # Initialize matching transformer
    tfidf = TfidfTransformer()

    tfidf_fitted = tfidf.fit_fitTransform(

In [ ]:
songs_dir = Path("./songs")
for audio_file in sorted(songs_dir.rglob("*.wav")):
    print(audio_file)
    test_single_file(audio_file)

songs/Sonata in C Minor, D958/MIDI-Unprocessed_Schubert1-3_MID--AUDIO_05_R2_2018_wav.wav
Init 1/3 with method k-means++
Inertia for init 1/3: 6761744.5
Init 2/3 with method k-means++
Inertia for init 2/3: 6907520.0
Init 3/3 with method k-means++
Inertia for init 3/3: 6785958.0
[MiniBatchKMeans] Reassigning 161 cluster centers.
Minibatch step 1/1834: mean batch inertia: 535.7247314453125
[MiniBatchKMeans] Reassigning 5 cluster centers.
Minibatch step 2/1834: mean batch inertia: 647.4248657226562, ewa inertia: 647.4248657226562
Minibatch step 3/1834: mean batch inertia: 532.8087158203125, ewa inertia: 634.9277206978998
Minibatch step 4/1834: mean batch inertia: 503.5457763671875, ewa inertia: 620.6025212095699
Minibatch step 5/1834: mean batch inertia: 536.8676147460938, ewa inertia: 611.4725053208833
Minibatch step 6/1834: mean batch inertia: 494.53662109375, ewa inertia: 598.7224286080489
Minibatch step 7/1834: mean batch inertia: 479.8856506347656, ewa inertia: 585.7650883914016
Minib

In [ ]:
file1 = 
file2 = 

